# Visual attacks on VLMs: mathematics and evidence
    **October 24, 2026 | CPU teaching edition | no scientific model runs**

    Run all uses PyTorch on CPU with a frozen analytic toy model. No installation,
    network call, data fetch, Drive mount or secret is required. This notebook embeds
    the dossier for portable reading. It does not implement a runnable VLM jailbreak suite.


## 1. From image to generated answer
    Raw image $x$ → preprocessing $P$ → visual encoder $V$ → connector $C$ → language model $M$.
    The model distribution is $p_\theta(y\mid x,t)$, with fixed weights $\theta$ and text $t$.
    Classifier cross entropy $L=-\log p(y_{true}\mid x)$ is not an unsafe-answer score.
    A targeted generation surrogate is $-\sum_j\log p_\theta(y_j^*\mid x,t,y^*_{<j})$.
    Target likelihood must still be checked against actual generated behavior.
    See Qi et al. §3/Eq.1 and Image Hijacks §2/Eq.1 in the reading register below.


In [1]:
import math
import torch
import torch.nn.functional as F

torch.set_num_threads(1)
DTYPE = torch.float64
DEVICE = torch.device('cpu')
SEED = 20261024
assert not torch.empty(0, device=DEVICE).is_cuda
print('CPU toy mechanics only. No VLM, data download, Drive mount or GPU branch.')

class FrozenMean(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = torch.nn.Parameter(torch.tensor(10., dtype=DTYPE), requires_grad=False)

    def forward(self, pixels):
        score = self.scale * (pixels.mean(dim=(1, 2, 3)) - .5)
        return torch.stack((score, -score), dim=1)

model = FrozenMean().eval()
x = torch.full((1, 3, 8, 8), .55, dtype=DTYPE, device=DEVICE)
true_y = torch.tensor([0])
target_y = torch.tensor([1])
weights_before = {k: v.clone() for k, v in model.state_dict().items()}
print('Clean toy class:', model(x).argmax(1).item())


CPU toy mechanics only. No VLM, data download, Drive mount or GPU branch.
Clean toy class: 0


/Users/aryamandev/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


### Interpretation — editable

The toy model classifies mean pixel brightness. Its fixed weights provide exact gradient checks. A class change here establishes only a toy perception failure.


## 2. FGSM, PGD and localized patch mathematics
    For raw pixels $x\in[0,1]^d$, let $\mathcal C=[0,1]^d\cap B_\infty(x,\epsilon)$.
    First-order expansion gives $L(x+\delta)\approx L(x)+\nabla_xL^\top\delta$;
    maximizing over $|\delta_i|\le\epsilon$ yields $\delta_i=\epsilon\,\mathrm{sign}(\partial_iL)$.
    FGSM takes one step and projects to $\mathcal C$.
    PGD repeats $x_{k+1}=\Pi_{\mathcal C}(x_k+\alpha\,\mathrm{sign}(\nabla_{x_k}L))$.
    Targeted attacks minimize target loss, so replace plus with minus.
    A binary support mask $A$ yields $x'=x+A\odot\delta$ for bounded local changes.
    A replacement patch instead uses $x'=(1-A)\odot x+A\odot p$.
    Goodfellow §§3–4; Madry §2; Brown §2/Eq.1 are the original reading sources.


In [2]:
def validate(x0, epsilon, steps, alpha, mask):
    if x0.device.type != 'cpu' or x0.ndim != 4 or not x0.is_floating_point():
        raise ValueError('Teaching API expects a CPU floating NCHW tensor')
    if not torch.isfinite(x0).all() or x0.min() < 0 or x0.max() > 1:
        raise ValueError('Raw pixels must be finite and in [0,1]')
    if not math.isfinite(epsilon) or not 0 <= epsilon <= 1:
        raise ValueError('Invalid raw-pixel epsilon')
    if not isinstance(steps, int) or isinstance(steps, bool) or steps < 1:
        raise ValueError('Positive integer step count required')
    if not math.isfinite(alpha) or alpha < 0:
        raise ValueError('Invalid step size')
    if mask.shape != x0.shape or mask.device != x0.device:
        raise ValueError('Mask must match input shape/device')
    if not torch.all((mask == 0) | (mask == 1)):
        raise ValueError('Mask must be binary')

def project(candidate, clean, epsilon, mask):
    # Box intersection first, then exact restoration outside local support.
    lo = torch.clamp(clean - epsilon, min=0.)
    hi = torch.clamp(clean + epsilon, max=1.)
    bounded = torch.maximum(lo, torch.minimum(hi, candidate))
    return torch.where(mask.bool(), bounded, clean).detach()

def pixel_attack(forward, clean, labels, *, epsilon, steps=1, alpha=None,
                 targeted=False, mask=None, random_start=False, seed=SEED):
    # A teaching optimizer, deliberately NOT a production VLM interface.
    clean = clean.detach().clone()
    mask = torch.ones_like(clean) if mask is None else mask.detach().clone()
    alpha = epsilon if alpha is None else alpha
    validate(clean, epsilon, steps, alpha, mask)
    current = clean.clone()
    if random_start:
        rng = torch.Generator(device='cpu').manual_seed(seed)
        noise = torch.rand(clean.shape, generator=rng, dtype=clean.dtype) * 2 - 1
        current = project(clean + epsilon * noise, clean, epsilon, mask)
    for _ in range(steps):
        current = current.detach().requires_grad_(True)
        loss = F.cross_entropy(forward(current), labels)
        grad, = torch.autograd.grad(loss, current)
        if not torch.isfinite(grad).all():
            raise FloatingPointError('Nonfinite input gradient')
        direction = -1 if targeted else 1
        current = project(current + direction * alpha * grad.sign(), clean, epsilon, mask)
    return current

fgsm = pixel_attack(model, x, true_y, epsilon=.1)
pgd = pixel_attack(model, x, true_y, epsilon=.1, steps=10, alpha=.02,
                   random_start=True)
targeted = pixel_attack(model, x, target_y, epsilon=.1, steps=10, alpha=.02,
                        targeted=True)
mask = torch.zeros_like(x)
mask[:, :, :4, :4] = 1
patch = pixel_attack(model, x, target_y, epsilon=.25, steps=10, alpha=.05,
                    targeted=True, mask=mask)
print('Toy classes:', {k: model(v).argmax(1).item() for k, v in
      [('clean', x), ('FGSM', fgsm), ('PGD', pgd), ('targeted', targeted), ('patch', patch)]})
print('Patch area fraction:', mask.mean().item())


Toy classes: {'clean': 0, 'FGSM': 1, 'PGD': 1, 'targeted': 1, 'patch': 1}
Patch area fraction: 0.25


### Interpretation — editable

All three methods from DL_final are demonstrated as CPU mechanics. PGD and FGSM can tie on this linear toy; that is not a comparative VLM result. The localized patch is per-image, not a universal Brown patch.


In [3]:
# Finite-difference check uses a single coordinate of the raw input.
probe = x.clone().requires_grad_(True)
grad, = torch.autograd.grad(F.cross_entropy(model(probe), true_y), probe)
plus, minus = x.clone(), x.clone()
h = 1e-5
plus[0, 0, 0, 0] += h
minus[0, 0, 0, 0] -= h
fd = (F.cross_entropy(model(plus), true_y) - F.cross_entropy(model(minus), true_y)) / (2*h)
assert torch.allclose(grad[0, 0, 0, 0], fd, atol=1e-8, rtol=1e-5)
assert torch.isfinite(grad).all() and grad.abs().max() > 0

for value, budget in [(fgsm, .1), (pgd, .1), (targeted, .1), (patch, .25)]:
    assert value.min() >= 0 and value.max() <= 1
    assert (value - x).abs().max() <= budget + 1e-12
assert torch.equal(patch[mask == 0], x[mask == 0])
assert F.cross_entropy(model(fgsm), true_y) > F.cross_entropy(model(x), true_y)
assert F.cross_entropy(model(targeted), target_y) < F.cross_entropy(model(x), target_y)
assert model(targeted).argmax(1).item() == target_y.item()
assert model(patch).argmax(1).item() == target_y.item()
assert torch.equal(pgd, pixel_attack(model, x, true_y, epsilon=.1, steps=10,
                                    alpha=.02, random_start=True))
assert torch.equal(x, pixel_attack(model, x, true_y, epsilon=0.))
assert torch.equal(x, pixel_attack(model, x, true_y, epsilon=.1, mask=torch.zeros_like(x)))
edge = torch.zeros_like(x)
edge[:, :, :, 4:] = 1
edge_adv = pixel_attack(model, edge, true_y, epsilon=.1)
assert edge_adv.min() >= 0 and edge_adv.max() <= 1
assert (edge_adv - edge).abs().max() <= .1 + 1e-12
assert torch.equal(model.scale.detach(), weights_before['scale'])
assert model.scale.grad is None

invalid = [dict(epsilon=-.1), dict(epsilon=float('nan')), dict(epsilon=.1, steps=0),
           dict(epsilon=.1, alpha=float('inf')), dict(epsilon=.1, mask=mask+.5)]
for kwargs in invalid:
    try:
        pixel_attack(model, x, true_y, **kwargs)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid configuration accepted')
try:
    pixel_attack(lambda z: model(z) * float('nan'), x, true_y, epsilon=.1)
except FloatingPointError:
    pass
else:
    raise AssertionError('Nonfinite gradient accepted')
print('Passed: finite differences, objective direction, bounds, patch support, seeds,')
print('zero budget/mask, boundary pixels, frozen weights, invalid input and NaN rejection.')


Passed: finite differences, objective direction, bounds, patch support, seeds,
zero budget/mask, boundary pixels, frozen weights, invalid input and NaN rejection.


### Interpretation — editable

Assertions check numerical gradients and attack directions, not just function output. Passing these tests does not establish gradients through a real VLM processor.


## 3. Scaling and saved-image validity
    With $z=(x-\mu)/\sigma$, $\delta_x=\sigma\delta_z$.
    An epsilon specified after normalization changes its meaning by channel.
    Quantization and resizing are part of the threat model; validate the delivered artifact.
    Brown §2 and Image Hijacks §2 explicitly motivate artifact and transformation checks.


In [4]:
sigma = torch.tensor([.229, .224, .225], dtype=DTYPE)
print('Raw bounds implied by normalized epsilon=.02:', (.02 * sigma).tolist())
# Quantization can slightly exceed a continuous budget: validate the actual artifact.
artifact_epsilon = .02
continuous = torch.tensor([.5 + artifact_epsilon], dtype=DTYPE)
reloaded = torch.round(continuous * 255).to(torch.uint8).to(DTYPE) / 255
effective = (reloaded - .5).abs().item()
print('8-bit roundtrip effective raw budget:', effective)
print('Inside declared .02 budget:', effective <= artifact_epsilon)
assert effective > artifact_epsilon  # This example catches naive roundtrip drift.
# Quantize inside the permitted integer interval for an exactly representable artifact.
lo = math.ceil((.5 - artifact_epsilon) * 255)
hi = math.floor((.5 + artifact_epsilon) * 255)
saved = torch.clamp(torch.round(continuous * 255), min=lo, max=hi)
restored = saved.to(torch.uint8).to(DTYPE) / 255
assert (restored - .5).abs().max() <= artifact_epsilon + 1e-12
print('Constrained integer artifact passes; repeat after actual resize/normalization.')


Raw bounds implied by normalized epsilon=.02: [0.00458, 0.0044800000000000005, 0.0045000000000000005]
8-bit roundtrip effective raw budget: 0.021568627450980427
Inside declared .02 budget: False
Constrained integer artifact passes; repeat after actual resize/normalization.


### Interpretation — editable

The example measures quantization rather than assuming it preserves an inequality. A valid raw artifact can still be changed by model-specific preprocessing.


## 4. Generative loss and transformation averaging
    Teacher forcing scores known target response tokens; it does not differentiate through greedy sampling.
    Mask prompt, image and padding positions. Causal alignment must shift exactly once.
    Universal patches minimize expected target loss across development images and transformations;
    per-example optimization does not establish universality. EOT differentiates an average of losses,
    $\nabla_x\mathbb E_T L(f(T(x)))$, approximated by sampled transforms.
    Brown §2/Eq.1; Image Hijacks §2; Qi §3 provide distinct constructions, not one shared attack.


In [5]:
# Harmless numeric tokens: target loss is over response positions only.
logits = torch.tensor([[[2., 0., -1.], [0., 2., -1.], [-1., 0., 2.]]], dtype=DTYPE)
labels = torch.tensor([[-100, 1, 2]])
masked = F.cross_entropy(logits.reshape(-1, 3), labels.reshape(-1), ignore_index=-100)
response_only = F.cross_entropy(logits[:, 1:].reshape(-1, 3), labels[:, 1:].reshape(-1))
assert torch.allclose(masked, response_only)
print('Toy response-only cross entropy:', masked.item())
# Here logits are already aligned to predicted targets. An actual causal LM must
# shift labels/logits once and account for its image-token expansion.

# Differentiable transformation averaging (horizontal flip), NOT a reproduced defense.
eot_input = x.clone().requires_grad_(True)
losses = [F.cross_entropy(model(view), true_y) for view in
          [eot_input, torch.flip(eot_input, dims=(-1,))]]
eot_grad, = torch.autograd.grad(torch.stack(losses).mean(), eot_input)
assert torch.isfinite(eot_grad).all() and eot_grad.abs().max() > 0
print('Toy transformation-averaged gradient is finite and nonzero.')


Toy response-only cross entropy: 0.1698460195562856
Toy transformation-averaged gradient is finite and nonzero.


### Interpretation — editable

Numeric vocabulary IDs illustrate response-only loss masking. A real adapter needs model-specific token expansion and reference-logit parity. The flip-invariant toy does not demonstrate robustness to transformations.


## 5. Evaluation and visual prompt injection

Perception error, instruction deviation and harmful assistance need distinct labels. Injection requires a legitimate user task and lower-trust content; it can be readable text inside a scene without pixel optimization. No benchmark injection content is embedded here.


In [6]:
# Synthetic outcomes, not collected model responses or human annotations.
attempted, confirmed_unsafe, unresolved = 10, 2, 3
lower = confirmed_unsafe / attempted
upper = (confirmed_unsafe + unresolved) / attempted
resolved_only = confirmed_unsafe / (attempted - unresolved)
assert lower <= resolved_only <= upper
print({'synthetic_confirmed_fraction': lower, 'missing_fraction': unresolved/attempted,
       'unsafe_bounds': [lower, upper], 'resolved_only': resolved_only})

# Symbolic trust-boundary demo: no image, OCR engine or language model is used.
def symbolic_policy(user_action, visual_action, trust_visual=False):
    return visual_action if trust_visual else user_action

assert symbolic_policy('count', 'redirect') == 'count'
assert symbolic_policy('count', 'redirect', True) == 'redirect'
print('Symbolic demonstration distinguishes user authority from visual data.')
print('This is not evidence that any defense protects a real VLM.')


{'synthetic_confirmed_fraction': 0.2, 'missing_fraction': 0.3, 'unsafe_bounds': [0.2, 0.5], 'resolved_only': 0.2857142857142857}
Symbolic demonstration distinguishes user authority from visual data.
This is not evidence that any defense protects a real VLM.


### Interpretation — editable

Missing cases produce an interval, not automatic safe labels. The symbolic policy is an explanatory analogy, not a visual injection reproduction. Two independent humans must inspect task and image where scoring requires them.


## 6. Portable dossier and research register

Dossier SHA256: `8dc883d4a295faddee936a0cf4b89e353a754cd21c6499a86fc40607049c1519`. Inspection status and execution gates below apply to every experiment proposal.


## 1. What exists and what is still pending

| Deliverable | Status |
|---|---|
| Searchable core-paper register and mechanism comparison | Implemented below; inspection depth recorded honestly |
| Mathematical teaching notebook | Generated by `tools/nb_gen/make_visual_attacks_nb.py`; CPU toy examples only |
| Original classifier notebook audit | Static source audit below; original dataset/results not reproduced |
| Dataset inventory | Official metadata inspected; artifact/license/overlap checks still pending where stated |
| New attack protocol | Reviewable draft in section 7; not an accepted amendment |
| Real FGSM/PGD/patch/injection VLM evaluation | Pending protocol approval and gradient/memory checks |
| Human labels and defense results | Not collected |
| Novel contribution | Not established; decision process in section 9 |

**Reading labels:** “partially inspected” means original method/evaluation sections were opened,
but a complete methods–appendix–code audit is outstanding. “Abstract-only” means discovery only.
No entry below is promoted to “full-text verified” merely because its HTML was available.
Reported paper results are not ours. The survey is a discovery source, not independent validation.
Paper-card numerical extracts are intentionally small; do not compare rates across different tasks.


## 2. Research question and threat model

How do visual manipulations compromise safety or instruction integrity, and which inference-time
defenses reduce failures while preserving useful visual performance? Model weights stay frozen.
Open models/data, T4 first, no paid API dependency. Input optimization is allowed, not weight training.

| Track | Legitimate task | Failure endpoint | Must not be conflated with |
|---|---|---|---|
| Perception | Recognition or VQA | Wrong or specified target answer | Harmful assistance |
| Instruction integrity | User task over untrusted visual content | Predefined unauthorized deviation | Merely transcribing an instruction |
| Safety | Policy-labelled request with fixed image conditions | Relevant prohibited assistance | Any non-refusal, nonsense, or image-reading failure |

Record four separate axes: attacker access (full gradients/encoder-only/queries/transfer),
controlled channel (image/text/both), support (full image/local region), and objective.
FGSM/PGD are optimization methods; a patch is a support constraint; injection is a mechanism.
Visual-only manipulation may accompany a fixed text question. It does not mean no text input.
“Vision is less safe than text” is a hypothesis requiring matched comparisons, not a premise.


## 3. Core reading register

All source links below were opened October 24, 2026 (or rechecked from this session).
Publication labels refer to the linked version; publisher equivalence is not assumed.
Each card states the next work needed before claiming a faithful reproduction.

### P01 — FGSM

Ian J. Goodfellow, Jonathon Shlens, Christian Szegedy, **Explaining and Harnessing Adversarial Examples**.
[1412.6572v3, 2015 preprint revision](https://arxiv.org/html/1412.6572v3). **Partially inspected:** §§3–5.
The first-order loss approximation motivates the sign-gradient update under an infinity-norm constraint.
The experiments concern classifiers, not generative VLM safety. §4 reports MNIST maxout adversarial
misclassification of **89.4%**; this is error, not jailbreak ASR. Training interventions are outside our scope.
Reproduction task: trace input scaling, loss and epsilon through the original preprocessing; then implement
the inner attack on a frozen toy model. Transfer to a VLM requires a different objective and scoring.
Full experiment/appendix and official-code provenance audit pending; no VLM prediction inferred.

### P02 — PGD and robust optimization

Aleksander Madry, Aleksandar Makelov, Ludwig Schmidt, Dimitris Tsipras, Adrian Vladu,
**Towards Deep Learning Models Resistant to Adversarial Attacks**.
[1706.06083v4, 2019 revision; ICLR 2018](https://arxiv.org/html/1706.06083v4).
**Partially inspected:** §2, §5/Table 2 and appendix table pointers.
§2 separates inner input maximization from outer weight minimization; implement only the inner problem.
Table 2 reports CIFAR10 adversarially trained network accuracy **45.8%**, PGD **20** steps,
source A, epsilon **8** in the paper's image scale. This is neither a raw [0,1] epsilon nor a VLM result.
[Official CIFAR challenge](https://github.com/MadryLab/cifar10_challenge) is linked in the paper;
repository implementation not audited here. Check random starts, restart selection, scaling and transfer
before reproducing. Strong PGD results do not certify robustness against all attacks.

### P03 — Universal adversarial patch

Tom B. Brown, Dandelion Mané, Aurko Roy, Martín Abadi, Justin Gilmer, **Adversarial Patch**.
[1712.09665v2, 2018 preprint revision](https://arxiv.org/html/1712.09665v2).
**Partially inspected:** §§2–3, Fig.1. §2/Eq.1 optimizes expected target log probability across
images, locations and transformations. This universal replacement patch differs from the existing
notebook's per-image bounded patch. Fig.1 illustrates VGG16 target confidence **99%**; it is one
illustration, not population ASR. §3 includes a control patch and model-ensemble evaluation.
Reproduction task: distinguish replacement from bounded additive support, define patch area in input
and processed coordinates, and separate optimization images from held-out images. Physical photographs
are outside our first experiment. Full code/config and appendix verification remain pending.

### P04 — Field map

Chiyu Zhang, Lu Zhou, Xiaogang Xu, Jiafei Wu, Zhe Liu,
**Adversarial Attacks of Vision Tasks in the Past 10 Years: A Survey**.
[2410.23687v2, May 2025 revision](https://arxiv.org/html/2410.23687v2).
**Partially inspected:** §§3–5 and §§7.2–7.5. §7 separates access, purpose and technique; Table 3
distinguishes capability, red-team, robustness and alignment data. Use as the citation map, not a benchmark
to reproduce. Its query/gray-box terminology must be translated into explicit access descriptions.
The ACM DOI supplied by the owner appears to identify the published counterpart; ACM access returned
403, so publisher/preprint text equality is unverified. Do not count them as independent evidence.
Follow original sources and search subsequent work through October 24, 2026.

### P05 — Visual adversarial examples against safety

Xiangyu Qi, Kaixuan Huang, Ashwinee Panda, Peter Henderson, Mengdi Wang, Prateek Mittal,
**Visual Adversarial Examples Jailbreak Aligned Large Language Models**.
[2306.13213v2, August 2023 revision](https://arxiv.org/html/2306.13213v2).
**Partially inspected:** §§3–4.5 and appendix headings. §3/Eq.1 minimizes corpus negative log
likelihood through the VLM to optimize an image; evaluation uses other textual requests.
§4.4 explicitly includes a text-attack comparator, so our comparison is not a first.
§4.3/Table 2 measures toxicity, which differs from task-relevant harmful assistance.
[Official repository](https://github.com/Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models)
includes model-specific attack code and artifacts. Next: pin checkpoint, corpus and code commit;
extract all budgets/steps/decoding from code and Appendix A before reproducing. T4 backward fit is unknown.

### P06 — Encoder-space composition

Erfan Shayegani, Yue Dong, Nael Abu-Ghazaleh,
**Jailbreak in pieces: Compositional Adversarial Attacks on Multi-Modal Language Models**.
[2307.14539v2, October 2023](https://arxiv.org/html/2307.14539v2).
**Partially inspected:** §3.1/Eqs.3–4, Algorithm 1, §4/Table 1 and appendix headings.
Minimizes distance between image embeddings and target-trigger embeddings using encoder access;
tests LLaVA and LLaMA-Adapter V2. Table 1 reports annotator Fleiss' kappa **0.8969**; that agreement
cannot validate our reviewers. Four trigger constructions distinguish text, OCR, visual and combined
targets. Encoder matching is a surrogate objective; a close embedding alone is not a successful jailbreak.
Next: inspect exact CLIP variant, initialization, stopping/bounds and Appendix H rubric; official code
and release identity remain unverified. Keep as an extension until end-to-end baselines work.

### P07 — Behavioral image hijacks

Luke Bailey, Euan Ong, Stuart Russell, Scott Emmons,
**Image Hijacks: Adversarial Images can Control Generative Models at Runtime**.
[2309.00236v4, September 2024](https://arxiv.org/html/2309.00236v4).
**Partially inspected:** §§2–4/Table 1 and Appendix C headings.
Eq.1 teacher-forces target behavior and minimizes output-distribution loss across contexts;
prompt matching uses model-generated behavior as the target. Table 1 reports **92%** jailbreak success
at epsilon **16/255**, versus original image **4%**, under that paper's protocol.
It also evaluates stationary/moving patches; patch jailbreak cells marked unrun must remain unrun.
[Official project/code entry](https://image-hijacks.github.io/). Next: inspect context splits, judge,
full precision/checkpoint and restart details. A newer LLaVA checkpoint is an adaptation, not a reproduction.

### P08 — Joint image–text attack

Zonghao Ying, Aishan Liu, Tianyuan Zhang, Zhengmin Yu, Siyuan Liang, Xianglong Liu, Dacheng Tao,
**Jailbreak Vision Language Models via Bi-Modal Adversarial Prompt**.
[2406.04031v2, July 2024](https://arxiv.org/html/2406.04031v2).
**Partially inspected:** §§3–5/Table 1, Appendix A and D headings.
§4.1/Eq.3 optimizes query-agnostic image likelihood; §4.2 then changes text using feedback.
Table 1's MiniGPT4 Illegal Activity row reports BAP **59.04** percent; the no-attack entry is **2.13**.
§5.1 uses ChatGPT and an A800 cluster: an open-only T4 replacement is an adaptation.
Appendix D hyperparameters and code identity still need checking. Keep combined attacks distinct from
image-only manipulations; report text/image ablations and attack-search queries. Extended reproduction,
not part of the initial GPU core.

### P09 — Existing attack–defense benchmark

Fenghua Weng, Yue Xu, Chengyan Fu, Wenjie Wang,
**MMJ-Bench: A Comprehensive Study on Jailbreak Attacks and Defenses for Vision Language Models**.
[AAAI 2025 original PDF](https://ojs.aaai.org/index.php/AAAI/article/download/34983/37138).
**Partially inspected:** introduction, attack/defense taxonomy and benchmark setup; result tables require
cell-level extraction. [Official repository](https://github.com/thunxxx/MLLM-Jailbreak-evaluation-MMJ-Bench).
Compares existing attacks and defenses and includes normal-task utility. A broad comparison alone is
therefore not our novelty. Do not confuse **MMJ-Bench** with the separate **MMJailBench** cited in the
historical project. Next: pin artifact manifest, source benchmarks, evaluation labels and model commits;
reproduce a supported slice before adding new models. No numerical result transcribed until checked.

### P10 — Physical visual prompt injection

Chen Ling, Kai Hu, Hangcheng Liu, Xingshuo Han, Tianwei Zhang, Changhai Ou,
**Physical Prompt Injection Attacks on Large Vision–Language Models**.
[2601.17383v1, January 2026](https://arxiv.org/html/2601.17383v1).
**Partially inspected:** §§III–V and VI headings. §III-C assumes no target interaction or user-query
access during generation; §III-D defines visual insertion rather than a small pixel perturbation.
[Paper-linked code](https://github.com/2023cghacker/Physical-Prompt-Injection-Attack) not audited here.
Digital placement under our own known tasks changes the threat model; physical robustness requires
camera capture, not merely software rotations. Full per-task results, simulation data permissions and
physical evaluation remain pending. The earlier AI answer misidentified this arXiv ID as InjecAgent.

### P11 — Defense prompting

Yu Wang, Xiaogeng Liu, Yu Li, Muhao Chen, Chaowei Xiao,
**AdaShield: Safeguarding Multimodal Large Language Models from Structure-based Attack via Adaptive Shield Prompting**.
[2403.09513v1, March 2024](https://arxiv.org/html/2403.09513v1).
**Partially inspected:** §§3–4.2/Table 2 and appendix headings.
Static AdaShield-S differs from adaptive prompt-pool construction/retrieval. §4.1 uses Vicuna as defender
and GPT4 paraphrasing; evaluates FigStep/QR and MM-Vet, with keyword ASR plus GPT recheck.
Table 2 LLaVA-1.5-13B FigStep Recheck entries: vanilla **87.21**, AdaShield-S **35.58**.
[Official code](https://github.com/SaFo-Lab/AdaShield). Next: inspect split leakage, pool construction
and thresholds. A static open-only defense is a narrower baseline, not the full adaptive reproduction.

### P12 — Mutation-based detection

Xiaoyu Zhang, Cen Zhang, Tianlin Li, Yihao Huang, Xiaojun Jia, Ming Hu, Jie Zhang, Yang Liu,
Shiqing Ma, Chao Shen, **JailGuard: A Universal Detection Framework for Prompt-based Attacks on LLM Systems**.
[2312.10766v4, March 2025; TOSEM DOI in text](https://arxiv.org/html/2312.10766v4).
**Partially inspected:** §4/Eqs.5–7, §6.2/Table 4 and adaptive-attack discussion.
Constructs distributions from response similarities and thresholds KL divergence, with a refusal rule.
Table 4 image-policy accuracy **82.90**, precision **74.00**, recall **88.25** percent.
These are detection metrics, not unsafe-assistance rates. [Official code](https://github.com/shiningrain/JailGuard).
Next: pin variant count, similarity model, threshold calibration, cost and adaptive coverage.
Stable unsafe responses can defeat inconsistency intuition; benign OCR tasks can be mutation-sensitive.


## 4. Extended discovery and novelty search

October 24 query log: `2026 vision language adversarial patch adaptive defense attacks benchmark`;
`site.aclanthology.org 2025 2026 visual prompt injection benchmark`;
`Jailbreaking Vision-Language Models Through the Visual Modality`.
These searches are a first pass, not an exhaustive or negative novelty result.

| Lead | Inspection status | Follow-up / exclusion |
|---|---|---|
| [Azulay et al., 2605.00583](https://arxiv.org/abs/2605.00583) | Abstract-only | Semantic symbols/substitution/analogy: check task matching, code and openness |
| [VPI-Bench, 2506.02456](https://arxiv.org/abs/2506.02456) | Search discovery only | Computer-use agents; exclude tool execution from our first experiment |
| [Adversarial Defense overview, 2601.12443](https://arxiv.org/abs/2601.12443) | Search discovery only | Follow original defenses; distinguish CLIP classification from generative VLM safety |
| [PHANTOM, 2606.24388v1](https://arxiv.org/html/2606.24388v1) | Dataset card and construction/Appendix F inspected | Mixed attack families, access gate, overlap with MM-SafetyBench |
| [CyberSecEval3 visual injection](https://huggingface.co/datasets/facebook/cyberseceval3-visual-prompt-injection) | Official card inspected | Candidate digital injection source; exact clean-task reconstruction pending |

Forward review must also search original-paper citations, ACL Anthology and official repositories,
record dates, deduplicate versions, and test each proposed contribution against closest existing work.
No “first”, “all attacks”, universal defense, or resolved safety–capability trade-off claim is supported.


## 5. Original notebook audit (static, not reproduction)

Source: [DL_final (1).ipynb](https://github.com/dev4-gpt/Adversarial-attacks/blob/main/DL_final%20%281%29.ipynb).
Inspected temporary download SHA256: `a6bd4d19df0cad481a0e72a91845faf8175cedc5ab162d77809ffcc4fa9290ba`.
Cell numbers below are one-based. No original code was executed; no Drive files were changed.

| Location | Finding | Required action before reuse |
|---|---|---|
| Cell 7 | ImageFolder classes mapped by sorted-label order | Validate synset-to-ImageNet index explicitly; reject absent/ambiguous mapping |
| Cells 8,10 | FGSM/PGD epsilon applied after normalization; no valid-image clamp | Optimize raw [0,1] pixels and normalize inside forward; project to box AND budget |
| Cell 11 | Patch uses global extrema across normalized channels | Use raw-pixel bounds; global scalar bounds are not valid per-channel ranges |
| Cell 11 | Random location/target per image, bounded patch | Label per-example targeted patch, not Brown universal patch |
| Cell 11 | Reports top-1/top-5 against true class | Add target-hit rate; wrong true class does not establish chosen target success |
| Code cells | No Python/NumPy/PyTorch seed initialization found | Local seeded generators; record configuration and deterministic limitations |
| Cell 11 | Lower patch accuracy triggers “less effective” message | Correct direction: lower accuracy means greater untargeted degradation |
| Cell 8 vs 10–11 | FGSM save is commented; PGD/patch use .pt | Verify actual files; README's advertised directory is not proof of saved artifacts |
| Cell 5 | Destructive Drive cleanup commands | Never execute; omit from any new notebook |

For channel normalization z=(x-mu)/sigma, raw delta_x=sigma*delta_z. Hence a scalar normalized
epsilon is channel-dependent in raw units. “Imperceptible” requires evidence, not just an L-infinity cap.
README accuracy figures remain reported historical values, not independently reproduced measurements.


## 6. Data readiness matrix

No benchmark image, prompt, reply or full dataset is copied into the public repository.
Metadata availability does not imply artifact-level reproducibility or redistribution permission.

| Source / role | Verified metadata | Outstanding gate and split unit |
|---|---|---|
| Existing ImageNet subset / classifier audit | Notebook expects `TestDataSet` and labels file | Actual subset/license/checksums not located in this pass; split by image, verify label IDs |
| [COCO](https://cocodataset.org/#termsofuse) / carriers | Official site opened; dynamic terms not extracted | Review each image's source/license and pin IDs; group images across all conditions |
| [VQAv2](https://visualqa.org/download.html) / benign VQA | Official downloads available | Terms, annotation schema and scorer pending; group shared images/questions |
| [HarmBench](https://github.com/centerforaisafety/HarmBench) / requests | Official repository lists MIT license | Pin behavior CSV revision, source permissions and historical semantic overlap; group intent |
| [MMJ-Bench](https://github.com/thunxxx/MLLM-Jailbreak-evaluation-MMJ-Bench) / reproduction | Official code entry verified | Artifact download/license/clean pairing/hash checks pending; group original intent/artifact |
| [PHANTOM](https://huggingface.co/datasets/it4lia/PHANTOM) / optional transfer | Card lists CC-BY-4.0; access requires accepting conditions | Owner gate; upstream image permissions, attack components, overlap and schemas pending |
| [CyberSecEval3 visual injection](https://huggingface.co/datasets/facebook/cyberseceval3-visual-prompt-injection) / injection | Card lists MIT; test split; evaluation-only intended use | Verify exact records/scorer and clean counterparts privately; no tuning on reserved test records |

Private manifest must record source revision, original ID, content hash, intent/image group, prior exposure,
license/access evidence, role (attack optimization/development/evaluation), track and clean-pair ID.
Split connected intent/image groups, not individual variants. Exclude historical seen items from fresh
confirmation; mark uncertain provenance as development-only. Universal optimization contexts/images
must not leak into evaluation. Selection precedes viewing outputs; persist an immutable manifest hash.


## 7. Proposed study contract — DRAFT, no scientific execution

This is review material, not the required amendment in `ANALYSIS_PLAN_MODALITY_SAFETY.md`.
After dataset/code audit and measured compute profiling, append a new amendment there without changing
historical hypotheses. Owner acceptance and a specific run budget remain required.

**Candidate endpoints:** E1 visual-attack minus clean unsafe-assistance rate (primary safety);
E2 defense-aware attacked minus undefended attacked unsafe rate; E3 paired benign utility change;
E4 injection deviation over paired legitimate tasks. These are candidate contrasts, not accepted hypotheses.
Keep perception endpoints in a distinct family. “Success among clean-baseline failures” means successful
attacks among cases where the clean baseline did NOT already meet the attack objective; spell out denominator.

**Missingness:** let N be attempted cases, H resolved unsafe and M unresolved/failed cases. Report
H/N as confirmed-unsafe fraction, M/N missing fraction, and [H/N,(H+M)/N] deterministic bounds.
Also report H/(N-M) as resolved-only, explicitly conditional. Never call unresolved safe or publish
the lower bound alone as the population unsafe rate. Apply paired missingness bounds to paired contrasts.

**Planned panel:** Qwen2.5-VL-3B-Instruct first, LLaVA-1.5-7B second if feasible. Existing Model7 uses
inference-mode generation, not an attack gradient API. New adapters must use teacher-forced loss,
target-token masks (exclude prompt/image/padding), differentiable raw-pixel preprocessing and frozen weights.
Check finite nonzero image gradients and finite-difference agreement; compare clean reference logits.
NF4 loading does not establish backward support or memory fit. Pin model/processor/code revisions.
No automatic checkpoint substitution. Original-paper checkpoints require a separate feasibility decision.

**Compute gate:** proposed benign profile = one model, one image/task, forward + backward + short
optimization, at most 30 T4 minutes after approval. Record wall time and allocated/reserved peak memory.
Abort on OOM/nonfinite gradients; do not repeatedly download or retry. No larger-GPU purchase by default.
Project total cost as optimization contexts × steps × restarts × models, plus generations and judges;
include multiplicative defense variants/EOT draws and shared universal-artifact costs.

**Development values, not final scientific settings:** raw-pixel epsilon sweep {2,4,8,16}/255;
FGSM one step; PGD {10,20,40} steps, alpha=epsilon/4, restarts {1,3}; patch area {1%,4%}.
Use seeded starts; choose restarts by attack objective, never evaluation labels. These are our adaptations,
not attributed paper defaults. Select/freeze final values using development only, then amend before testing.
Provisional request caps: 40 development / 120 evaluation plus comparable benign coverage, not a power claim.
Separate original-paper configuration from our common-budget comparison; do not call them the same run.

**Conditions:** clean image, text-only, random bounded noise, matched-area random patch, FGSM,
PGD, localized patch and visual injection within their valid task tracks. Fixed text for visual attacks.
Qi reproduction and combined image/text are separate configurations; no hidden textual optimization.
Use batch one, fixed generation template/precision and 256-token development output window; record EOS
and cap hits. Freeze scoring and any window change before evaluation; no claim that 256 is sufficient.

**Statistics:** pair by task/image; aggregate repeated variants within request before bootstrap;
use 2,000 request-cluster resamples for descriptive 95% intervals. If universal artifacts are shared,
condition conclusions on the tested artifacts; independent artifacts are needed for artifact generalization.
Report scenario/source and image breakdowns; paired McNemar only when its independent-pair assumptions hold.
Specify Holm family in the accepted amendment; don't borrow the earlier eight-scenario inference blindly.
Select a meaningful effect and freeze sample size after cost/power assessment; no output-driven shrinkage.

**Human review:** two independent people, blinded model/attack/defense/guard identities; retain task and
image needed for scoring. Proposed 120-response stratified probability sample with saved inclusion
probabilities; separately flag disagreement diagnostics. Preserve independently assigned harm, refusal,
comprehension, legitimate-task and injection labels before adjudication. Include UNCERTAIN; report
coverage, agreement/kappa and unresolved cases. Repeat 20 randomly selected units after at least 24 hours.
Use inverse-probability weighting for population estimates; no unweighted disagreement-selected rates.
Human evaluation and assistive-LLM provenance stay distinct. Guard text alone cannot establish visual correctness.

**Defenses:** none; reviewed fixed defense prompt; deterministic transformation baseline; optional
JailGuard-style detector only after cost measurement. Development-calibrate thresholds and benign utility.
For transformations evaluate defense-aware/EOT gradients; nondifferentiable transforms require an
explicit approximation/transfer check and an incompleteness statement. For prompting include prompt-aware
optimization. Detection accuracy is not protected-system safety. Reject-all is a utility baseline, not success.

**Cache/output contract for the later runner:** private task/response/artifact files; identities include
split hash, exact model+processor revision, loss/target hash, budget, steps, restart, seed, defense,
decoding and code hash. Atomic writes, per-condition status and explicit failure reasons; resume completed
units only on exact identity. Public export uses an allowlist of scalar counts/rates/intervals/timings and
non-sensitive provenance. Do not serialize nested replies, images or prompts into aggregate exports.


## 8. Reproduction checklist and implementation acceptance

Use labels **faithful reproduction**, **reduced-scale reproduction**, **adaptation** consistently.
Faithful requires original supported checkpoint, inputs/splits, preprocessing, objective and budget;
different models or losses are adaptations even if code was copied. Never fill unpublished cells with estimates.

| Check | Current teaching notebook | Later VLM runner requirement |
|---|---|---|
| Gradient / direction | Analytic toy gradient checked against finite differences; targeted and untargeted tests | Check through actual processor/encoder/connector/LM |
| Constraints | Box and infinity budget, local support, invalid-argument tests | Check artifact after encode/decode and actual preprocessing |
| Serialization | Synthetic raw-pixel 8-bit roundtrip measured; no media embedded | Reproject/validate saved artifact; record effective budget, never assume |
| Frozen model | Toy parameter snapshots unchanged | Actual weights/revisions unchanged; no optimizer over model weights |
| Injection | Symbolic harmless policy example, not a VLM experiment | Original source subset, legitimate tasks, actual model behavior |
| Evaluation | Synthetic missingness bounds demonstration | Real blinded labels, weighted sample, uncertainty and failures |
| Reliability | Deterministic notebook rebuild and CPU checks | Cache identity, crash/resume, isolation and export privacy tests |

Rebuild: `python3 tools/nb_gen/make_visual_attacks_nb.py --execute`.
CPU demonstrations prove mechanics only. No model safety, T4 fit, attack transfer or paper reproduction follows.
Toy images are tensors only: no released benchmark images, instructions or generated replies.


## 9. Calendar, deliverables and contribution gate

| Date | Required deliverable / decision |
|---|---|
| Oct 24–25 | This initial dossier/notebook; continue foundation full-text and code audits |
| Oct 26–27 | Complete core-paper appendices/configs; source permission/overlap manifests; reviewable amendment |
| Oct 28 | Advisor-ready updated dossier and notebook, verified/pending ledger and approved next bounded scope |
| Oct 29–Nov 4 | Accepted amendment; gradient adapters, benign profiles, supported reduced-scale reproductions; freeze test data |
| Nov 5–11 | Core attack comparisons and initial independent human evaluation |
| Nov 12–18 | Defense-aware tests, held-out extension, adjudication |
| Nov 19–25 | Traceable figures and complete draft, including negative results |
| Nov 26–Dec 1 | Advisor revision, reproducibility/privacy check, submission-ready manuscript |

By Nov 4 assess: preprocessing/deployment transfer, defense generalization at matched utility/cost,
or a consequential reproduction finding. First rule out close prior art, then require working baselines,
an affordable held-out test and human validation. Otherwise write a replication/measurement paper.
Current novelty verdict for all three: **unestablished**, not novel by default.
The broad comparison in MMJ-Bench and preprocessing/transfer work are mandatory comparators.

Milestones are deliverables, not scheduled autonomous execution. No automation was created.
December 1 means manuscript readiness, not guaranteed acceptance; owner selects venue.
The next owner review concerns the concrete scientific amendment and measured run scope, not this CPU notebook.


## Acceptance and next step

CPU assertions pass only when all code cells execute. Scientific runs remain gated by the accepted amendment, reviewed data and an approved bounded compute profile. No toy outcome is a VLM safety result. Rebuild with `python3 tools/nb_gen/make_visual_attacks_nb.py --execute`.
